In [ ]:
from itertools import product
from pathlib import Path
import re
from functools import reduce
from operator import or_, add
import json

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from skimage.measure import ransac
from skimage.transform import AffineTransform, SimilarityTransform, EuclideanTransform

from calmutils.descriptors import descriptor_local_qr, match_descriptors_kd
from calmutils.stitching.registration import register_iterative
from calmutils.stitching.transform_helpers import translation_matrix
from transform_helpers import affine_transform_nd

In [ ]:
def combine_dicts_along_keys(*dicts, combine_function=add, error_missing_key=False):

    """
    Combine the values for each key in multiple dicts via reduction with a user-specified function.
    By default, combines all present values for a key, skipping missing,
    but can also be set to raise error if a key is not present in all dicts.
    """

    # TODO: general-purpose function, move to CalmUtils?

    # get all keys present in any dict
    all_keys = reduce(or_, [d.keys() for d in dicts])
    combined_dicts = {}

    for k in all_keys:
        # get present values for key
        present_values = [d[k] for d in dicts if k in d]

        # number of present values does not match number of dicts -> raise error if desired
        if error_missing_key and (len(present_values) != len(dicts)):
            raise ValueError(f"key '{k}' not found in all dicts.")

        # combine via reduction with combine_function
        combined_value = reduce(combine_function, present_values)
        combined_dicts[k] = combined_value

    return combined_dicts

In [ ]:
detection_table1_path = '/Users/david/Desktop/gs781/merge_global_coords_dna.csv'
detection_table2_path = '/Users/david/Desktop/gs781/merge_global_coords_rna.csv'

coordinate_columns_yx = ["y_global_um", "x_global_um"]
coordinate_column_z = "z_global_um"

coordinate_columns_pixel = ["z", "y", "x"]

image_file_column = "img"
IMAGE_ID_COLUMN = "image_id"

# to determine coverslip plane, we get the z position of a low quantile of detections
# this should correspond to beads on the coverslip
z_bottom_quantile = 0.1

# descriptor and matching parameters
n_neighbors = 4
redundancy = 0
descriptor_match_ratio = 2

ransac_max_error = 4.0
ransac_max_trials = 100_000


In [ ]:
# combined column coords
coordinate_columns = [coordinate_column_z] + coordinate_columns_yx

df1 = pd.read_csv(detection_table1_path)
df2 = pd.read_csv(detection_table2_path)

len(df1[image_file_column].unique()), len(df2[image_file_column].unique())

In [ ]:
def get_file_stem_without_channel(path):
    """
    get the stem of a file path without ending and a suffix _ch{ch_id}.
    """

    # check for presence of channel ending and only remove if necessary
    stem = Path(path).stem
    if re.match(".*_ch[0-9]+", stem):
        file_id, ch_id = stem.rsplit("_", 1)
        return file_id
    else:
        return stem

# add cleaner image_id column (will be used in saved transforms as well)
df1[IMAGE_ID_COLUMN] = df1[image_file_column].apply(get_file_stem_without_channel)
df2[IMAGE_ID_COLUMN] = df2[image_file_column].apply(get_file_stem_without_channel)

## 0) Get pixel size and stage position transforms for all images.

Could also be done in *get global coordinates notebook* but we get them as explicit transformation matrices here.

In [ ]:
pixel_size_transforms = {}
stage_position_transforms = {}

# go over both dfs
for img_id, dfi in pd.concat([df1, df2]).groupby(IMAGE_ID_COLUMN):

    # get pixel and world coords
    pixel_coords = dfi[coordinate_columns_pixel].values
    world_coords = dfi[coordinate_columns].values

    # estimate affine transform (pixel size scale + stage translation)
    at = AffineTransform(dimensionality=3)
    at.estimate(pixel_coords, world_coords)

    # 2x diag (inner extracts diag, outer makes new diag matrix with only those entries)
    mat_pixelsize = np.diag(np.diag(at.params))

    mat_stage_translation = translation_matrix(at.params[:-1, -1])

    pixel_size_transforms[img_id] = AffineTransform(mat_pixelsize)
    stage_position_transforms[img_id] = AffineTransform(mat_stage_translation)

## 1) Get coverslip position, get shift to align

First, we estimate the coverslip position $z_{cs}$ in every image by getting a low quantile of all detections in that image.
A transformation that virtually aligns the coverslip positions is the translation $(-z_{cs}, 0, 0)$.

In [ ]:
z_transforms = {}

for (image_id, dfi) in df1.groupby(IMAGE_ID_COLUMN):
    z_coords = dfi[coordinate_column_z]
    z_transforms[image_id] = AffineTransform(translation_matrix([-np.quantile(z_coords, z_bottom_quantile), 0, 0]))

for (image_id, dfi) in df2.groupby(IMAGE_ID_COLUMN):
    z_coords = dfi[coordinate_column_z]
    z_transforms[image_id] = AffineTransform(translation_matrix([-np.quantile(z_coords, z_bottom_quantile), 0, 0]))

## 2) Global Alignment of two datasets via beads

In [ ]:
def get_transformed_coordinates(df, transforms=None, coordinate_columns=coordinate_columns, key=IMAGE_ID_COLUMN):

    # no transform to apply -> just return values of coordinate columns
    if transforms is None:
        return df[coordinate_columns].values

    coords_tr = []

    # go through all image_ids and apply corresponding transform from transform dict
    # NOTE: sort=False to keep same order as in original dataframe
    for image_id, dfi in df.groupby(key, sort=False):
        coords = dfi[coordinate_columns].values
        coords = transforms[image_id](coords)
        coords_tr.append(coords)
    return np.concatenate(coords_tr, axis=0)

In [ ]:
coords1 = get_transformed_coordinates(df1, z_transforms)
coords2 = get_transformed_coordinates(df2, z_transforms)

desc1, idx1 = descriptor_local_qr(coords1, n_neighbors, redundancy)
desc2, idx2 = descriptor_local_qr(coords2, n_neighbors, redundancy)

matches = match_descriptors_kd(desc1, desc2, max_ratio=1/descriptor_match_ratio)

len(matches)

In [ ]:
matched_idxs1 = idx1[matches[:, 0]]
matched_idxs2 = idx2[matches[:, 1]]

matched_coords1 = coords1[matched_idxs1]
matched_coords2 = coords2[matched_idxs2]

transform_global, inliers_global = ransac((matched_coords1, matched_coords2), EuclideanTransform, 4, ransac_max_error, max_trials=ransac_max_trials)
residuals = np.linalg.norm(transform_global(matched_coords1[inliers_global]) - matched_coords2[inliers_global], axis=1)

print(f"RANSAC inliers: {inliers_global.sum()} / {len(matched_coords1)}")
print(f"Residual error (mean, max): {residuals.mean() :.3f}, {residuals.max() :.3f}")

# use the global transform for each image in dataset1 (moving)
transforms_global = {image_id: transform_global for image_id in df1[IMAGE_ID_COLUMN].unique()}
# for dataset2 (target), append identity transform to have same number of transforms
transforms_global |= {image_id: AffineTransform(dimensionality=3) for image_id in df2[IMAGE_ID_COLUMN].unique()}

### Save

In [ ]:
json_save_path = '/Users/david/Desktop/gs781/transformations.json'

dict_to_save = {}
for img_id in pd.concat([df1, df2])[IMAGE_ID_COLUMN].unique():

    tr_info_psz = "pixel_size", pixel_size_transforms[img_id].params.ravel().tolist()
    tr_info_stage = "stage_position", stage_position_transforms[img_id].params.ravel().tolist()
    tr_info_z = "coverslip_align", z_transforms[img_id].params.ravel().tolist()
    tr_info_reg_global = "global_registration", transforms_global[img_id].params.ravel().tolist()

    dict_to_save[img_id] = [tr_info_psz, tr_info_stage, tr_info_z, tr_info_reg_global]

with open(json_save_path, "w") as fd:
    json.dump(dict_to_save, fd, indent=1)

In [ ]:
from matplotlib import pyplot as plt
import napari

tr_combined = combine_dicts_along_keys(z_transforms, transforms_global)

coords_tr1 = get_transformed_coordinates(df1, tr_combined)
coords_tr2 = get_transformed_coordinates(df2, tr_combined)

plt.scatter(*coords_tr1.T[1:], s=0.002, alpha=0.8)
plt.scatter(*coords_tr2.T[1:], s=0.002, alpha=0.8)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_points(coords_tr1[:, 0:], face_color='cyan', border_color="#FFF0", size=3)
viewer.add_points(coords_tr2[:, 0:], face_color='magenta', border_color="#FFF0", size=3)

## 3) Local Alignment

Now, we repeat the alignment as in step 2 on a per-image basis:

- we consider pairs of images from both datasets if their mean transformed coordinates differ by less than a threshold (~FOV size)
- For overlapping images, we perform descriptor matching and RANSAC
- the inlier point matches of all pairs with enough matches are used to calculate globally optimal consensus transforms (like in Multiview Reconstruction)

In [ ]:
# maximal distance of the mean coordinates of images to be still considered overlapping
overlap_mean_distance_cutoff = 50

# how many points have to match (and remain after RANSAC) to consider pair of images
min_matches_local = 12

redundancy_local = 1

max_error_local = 2.0

In [ ]:
from scipy.spatial import distance_matrix

# combine transforms so far
tr_combined = combine_dicts_along_keys(z_transforms, transforms_global)

# pre-apply to a copy of datasets
df1_with_tr = df1.copy()
df1_with_tr[[f"{c}_tr" for c in coordinate_columns]] = get_transformed_coordinates(df1_with_tr, tr_combined)
df2_with_tr = df2.copy()
df2_with_tr[[f"{c}_tr" for c in coordinate_columns]] = get_transformed_coordinates(df2_with_tr, tr_combined)

# get mean coords per image, select only pairs with difference less than cutoff
mean_coords_1 = df1_with_tr.groupby(IMAGE_ID_COLUMN)[[f"{c}_tr" for c in coordinate_columns]].mean().values
mean_coords_2 = df2_with_tr.groupby(IMAGE_ID_COLUMN)[[f"{c}_tr" for c in coordinate_columns]].mean().values

img_ids_1 = np.sort(df1[IMAGE_ID_COLUMN].unique())
img_ids_2 = np.sort(df2[IMAGE_ID_COLUMN].unique())
overlapping_fields = [(img_ids_1[i], img_ids_2[j]) for i,j in (np.argwhere(distance_matrix(mean_coords_1, mean_coords_2) < overlap_mean_distance_cutoff))]

In [ ]:
to_optimize_round1 = {}

for img_id_1, img_id_2 in tqdm(overlapping_fields):

    coords_tr_1 = df1_with_tr[df1_with_tr[IMAGE_ID_COLUMN] == img_id_1][[f"{c}_tr" for c in coordinate_columns]].values
    coords_tr_2 = df2_with_tr[df2_with_tr[IMAGE_ID_COLUMN] == img_id_2][[f"{c}_tr" for c in coordinate_columns]].values

    desc1, idx1 = descriptor_local_qr(coords_tr_1, n_neighbors, redundancy_local, scale_invariant=True)
    desc2, idx2 = descriptor_local_qr(coords_tr_2, n_neighbors, redundancy_local, scale_invariant=True)
    matches = match_descriptors_kd(desc1, desc2, max_ratio=1/2.0)

    if len(matches) < min_matches_local:
        continue

    coords_match_1 = coords_tr_1[idx1[matches[:, 0]]]
    coords_match_2 = coords_tr_2[idx2[matches[:, 1]]]

    model, inliers = ransac((coords_match_1, coords_match_2), EuclideanTransform, 3, max_error_local, max_trials=1000)

    if inliers is None or (inliers.sum() < min_matches_local):
        continue

    coords_inliers_1 = coords_match_1[inliers]
    coords_inliers_2 = coords_match_2[inliers]
    to_optimize_round1[(img_id_1, img_id_2)] = (coords_inliers_1, coords_inliers_2)

In [ ]:
refine_transforms_round1 = register_iterative(to_optimize_round1, transform_type=EuclideanTransform, max_iterations=500)
print(f"Per-tile transforms estimated for {len(refine_transforms_round1)} images")

# add identity transforms for the images for which we did not find transform
for img_id in pd.concat([df1, df2])[IMAGE_ID_COLUMN].unique():
    if img_id not in refine_transforms_round1:
        refine_transforms_round1[img_id] = AffineTransform(dimensionality=3)

### Save / Visualize

In [ ]:
json_save_path = '/Users/david/Desktop/gs781/transformations_1round.json'

dict_to_save = {}
for img_id in pd.concat([df1, df2])[IMAGE_ID_COLUMN].unique():

    tr_info_psz = "pixel_size", pixel_size_transforms[img_id].params.ravel().tolist()
    tr_info_stage = "stage_position", stage_position_transforms[img_id].params.ravel().tolist()
    tr_info_z = "coverslip_align", z_transforms[img_id].params.ravel().tolist()
    tr_info_reg_global = "global_registration", transforms_global[img_id].params.ravel().tolist()
    tr_info_tile_round1 = "tile_registration_round1", refine_transforms_round1[img_id].params.ravel().tolist()

    dict_to_save[img_id] = [tr_info_psz, tr_info_stage, tr_info_z, tr_info_reg_global, tr_info_tile_round1]

with open(json_save_path, "w") as fd:
    json.dump(dict_to_save, fd, indent=1)

In [ ]:
from matplotlib import pyplot as plt
import napari

tr_combined = combine_dicts_along_keys(z_transforms, transforms_global, refine_transforms_round1)

coords_tr1 = get_transformed_coordinates(df1, tr_combined)
coords_tr2 = get_transformed_coordinates(df2, tr_combined)

plt.scatter(*coords_tr1.T[1:], s=0.002, alpha=0.8)
plt.scatter(*coords_tr2.T[1:], s=0.002, alpha=0.8)

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_points(coords_tr1[:, 0:], face_color='cyan', border_color="#FFF0", size=3)
viewer.add_points(coords_tr2[:, 0:], face_color='magenta', border_color="#FFF0", size=3)

## 4) WIP: Second round (weak links)

In [ ]:
from calmutils.stitching.stitching import rectangle_corners
from itertools import combinations


def get_2nd_round_transforms(df, transforms):
    to_optimize_1 = {}
    fixed1 = set()

    for (img_file1, dfi1), (img_file2, dfi2) in combinations(df.groupby(image_file_column), 2):
        ci1 = dfi1[coordinate_columns].values.mean(axis=0)
        ci2 = dfi2[coordinate_columns].values.mean(axis=0)

        if np.linalg.norm(ci1 - ci2) > 50:
            continue

        c = rectangle_corners(np.ones(len(coordinate_columns)))

        if img_file1 in transforms and img_file2 in transforms:
            continue

        if img_file1 in transforms:
            to_optimize_1[(img_file1, img_file2)] = [transforms[img_file1](c), c]
            fixed1.add(img_file1)
        elif img_file2 in transforms:
            to_optimize_1[(img_file1, img_file2)] = [c, transforms[img_file2](c)]
            fixed1.add(img_file2)
        else:
            to_optimize_1[(img_file1, img_file2)] = [c, c]

    transforms_extra1 = register_iterative(to_optimize_1, fixed_indices=fixed1, transform_type=EuclideanTransform, max_iterations=500)
    return transforms_extra1


transforms_extra1 = get_2nd_round_transforms(df1, present_values)
transforms_extra2 = get_2nd_round_transforms(df2, present_values)


In [ ]:
coords_collected_1 = []
coords_collected_2 = []

for image_id, dfi in df1.groupby(image_file_column):
    tr = transform_global    
    if image_id in present_values:
        tr += present_values[image_id]
    if image_id in transforms_extra1:
        tr += transforms_extra1[image_id]

    coords = dfi[coordinate_columns].values
    coords_tr = (z_transforms[image_id] + tr)(coords)

    coords_collected_1.append(coords_tr)

for image_id, dfi in df2.groupby(image_file_column):
    tr = AffineTransform(dimensionality=3)
    if image_id in present_values:
        tr += present_values[image_id]
    if image_id in transforms_extra2:
        tr += transforms_extra2[image_id]

    coords = dfi[coordinate_columns].values
    coords_tr = (z_transforms[image_id] + tr)(coords)

    coords_collected_2.append(coords_tr)

coords_collected_1 = np.concatenate(coords_collected_1)
coords_collected_2 = np.concatenate(coords_collected_2)

from matplotlib import pyplot as plt
# plt.scatter(*coords_collected_1.T[1:], s=0.002, alpha=0.8)
plt.scatter(*coords_collected_2.T[1:], s=0.002, alpha=0.8)


In [ ]:
import napari

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
viewer.add_points(coords_collected_1[:, 0:], face_color='cyan', border_color="#FFF0", size=3)
viewer.add_points(coords_collected_2[:, 0:], face_color='magenta', border_color="#FFF0", size=3)

### OLD: other versions / tests for 3) Local Alignment

In [ ]:
from itertools import product
import warnings
from scipy.spatial import distance_matrix

# combine transforms so far
tr_combined = combine_dicts_along_keys(z_transforms, transforms_global)

# pre-apply to a copy of datasets
df1_with_tr = df1.copy()
df1_with_tr[[f"{c}_tr" for c in coordinate_columns]] = get_transformed_coordinates(df1_with_tr, tr_combined)
df2_with_tr = df2.copy()
df2_with_tr[[f"{c}_tr" for c in coordinate_columns]] = get_transformed_coordinates(df2_with_tr, tr_combined)

to_optimize_round1 = {}

for (img_file1, dfi1), (img_file2, dfi2) in tqdm(list(product(df1_with_tr.groupby(IMAGE_ID_COLUMN), df2_with_tr.groupby(IMAGE_ID_COLUMN)))):

    # get transformed coords
    coords_tr_1 = get_transformed_coordinates(dfi1)
    coords_tr_2 = get_transformed_coordinates(dfi2)

    # check 1: if difference of mean coords is too large, images don't overlap
    if np.linalg.norm(coords_tr_1.mean(axis=0) - coords_tr_2.mean(axis=0)) > 50:
        continue

    # check 2: closest points should be close
    min_coord_diff = distance_matrix(coords_tr_1, coords_tr_2).min()
    if min_coord_diff > 5.0:
        continue

    # get descriptors and match
    desc1, idx1 = descriptor_local_qr(coords_tr_1, n_neighbors, redundancy)
    desc2, idx2 = descriptor_local_qr(coords_tr_2, n_neighbors, redundancy)
    matches = match_descriptors_kd(desc1, desc2, max_ratio=1/2)

    # too few matches
    if len(matches) < 4:
        continue

    kp1 = coords_tr_1[idx1[matches[:,0]]]
    kp2 = coords_tr_2[idx2[matches[:,1]]]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore") # ignore warning about no inliers
        # NOTE: we ignore the RANSAC estimated model -> consensus model will be found in global opt.
        _, inliers_local = ransac((kp1, kp2), EuclideanTransform, 4, 4.0, max_trials=1_000)

    # RANSAC did not find model or very few inliers / low inlier ratio 
    if inliers_local is None or (sum(inliers_local) < 4) or (sum(inliers_local) / len(inliers_local) < 0.2):
        continue

    # add matched and inlier-filtered keypoints to global optimization input
    to_optimize_round1[(img_file1, img_file2)] = (kp1[inliers_local], kp2[inliers_local])


In [ ]:
aa = df1.iloc[matched_idxs1[inliers_global]]
bb = df2.iloc[matched_idxs2[inliers_global]]

aa = df1[[IMAGE_ID_COLUMN] + coordinate_columns].iloc[matched_idxs1[inliers_global]].reset_index(drop=True)
aa.columns = [c+"_1" for c in aa.columns]
bb = df2[[IMAGE_ID_COLUMN] + coordinate_columns].iloc[matched_idxs2[inliers_global]].reset_index(drop=True)
bb.columns = [c+"_2" for c in bb.columns]

inlier_df = pd.concat([aa, bb], axis=1)
inlier_df


In [ ]:
import numpy as np

tr_combined = combine_dicts_along_keys(z_transforms, transforms_global)

matched_tile_pairs = {}
for (img_file1, img_file2), dfi in inlier_df.groupby([IMAGE_ID_COLUMN + "_1", IMAGE_ID_COLUMN + "_2"]):

    if len(dfi) < 3:
        continue

    coords_1_tile = dfi[[c + "_1" for c in coordinate_columns]].values
    coords_2_tile = dfi[[c + "_2" for c in coordinate_columns]].values

    coords_1_tile = tr_combined[img_file1](coords_1_tile)
    coords_2_tile = tr_combined[img_file2](coords_2_tile)

    matched_tile_pairs[(img_file1, img_file2)] = (coords_1_tile, coords_2_tile)

refine_transforms_round1 = register_iterative(matched_tile_pairs, transform_type=EuclideanTransform, max_iterations=500)
print(f"Per-tile transforms estimated for {len(refine_transforms_round1)} images")

# add identity transforms for the images for which we did not find transform
for img_id in pd.concat([df1, df2])[IMAGE_ID_COLUMN].unique():
    if img_id not in refine_transforms_round1:
        refine_transforms_round1[img_id] = AffineTransform(dimensionality=3)

### OLD: multithreaded descriptors, etc.

Should no longer be necessary due to optimized descriptor code

In [ ]:
descriptor_cache = {}

for (tile_idx1, dfi1), (tile_idx2, dfi2) in tqdm(product(df1.groupby("img"), df2.groupby("img"))):
    pass

    coords1 = dfi1[list("zyx")].values
    coords2 = dfi2[list("zyx")].values

    if tile_idx1 in descriptor_cache:
        desc1, idx1 = descriptor_cache[tile_idx1]
    else:
        desc1, idx1 = descriptor_local_qr(coords1, 3, 1)
        descriptor_cache[tile_idx1] = (desc1, idx1)

    if tile_idx2 in descriptor_cache:
        desc2, idx2 = descriptor_cache[tile_idx2]
    else:
        desc2, idx2 = descriptor_local_qr(coords2, 3, 1)
        descriptor_cache[tile_idx2] = (desc2, idx2)

    matches = match_descriptors_kd(desc1, desc2, max_ratio=1/3)

    if len(matches) > 100:
        break



In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _get_matches(dfi1, dfi2):

    coords1 = dfi1[list("zyx")].values
    coords2 = dfi2[list("zyx")].values

    desc1, idx1 = descriptor_local_qr(coords1, 3, 1)
    desc2, idx2 = descriptor_local_qr(coords2, 3, 1)

    matches = match_descriptors_kd(desc1, desc2, max_ratio=1/3)

    return matches

with ThreadPoolExecutor() as pool:

    futures = []

    for (tile_idx1, dfi1), (tile_idx2, dfi2) in tqdm(product(df1.groupby("img"), df2.groupby("img"))):
        futures.append(pool.submit(_get_matches, dfi1, dfi2))
    
    for future in tqdm(futures):
        future.result()


In [ ]:
matched_coords_1 = coords1[idx1[matches[:,0]]]
matched_coords_2 = coords2[idx2[matches[:,1]]]

tr, inliers = ransac((matched_coords_1, matched_coords_2), SimilarityTransform, 3, 2, max_trials=1_000)
tr, inliers